# diagnostic prompt iteration

scratch notebook. **WIP.** keeping it ugly on purpose — the agent eval suite (week 9) is the source of truth, this is just where prompts get tried before they go in.

leaving the failed attempts in. easier to remember why v3 won than to rederive it later.

> **note to self:** do not refactor this. the messiness is the point.

In [ ]:
from sentinel.agent.llm import LLMClient, MockLLMClient
from pydantic import BaseModel, Field
import os, json

In [ ]:
# fake incident payload to iterate against. eventually load fixtures from
# tests/fixtures/incidents/*.json once those exist.
incident = {
    "asset_key": "bronze/tlc_yellow",
    "partition_key": "2024-04-01",
    "error_type": "HTTPStatusError",
    "error_message": "503 Server Error: Service Unavailable for url: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-04.parquet",
    "recent_metadata": {
        "run_id": "abc-123",
        "job_name": "__ASSET_JOB",
        "tags": {"dagster/partition": "2024-04-01"}
    },
    "upstream_lineage": [],
}

# uncomment to use real groq. mock by default so this notebook is reproducible offline.
# llm = LLMClient()
llm = MockLLMClient()

## v1: free-form, terrible

first attempt. groq llama-3.1-8b padded the response with a 4-paragraph preamble and used markdown headings. unparseable downstream.

In [ ]:
v1_prompt = f"""You are a senior data engineer. The following data pipeline asset failed:

{json.dumps(incident, indent=2)}

Diagnose the failure and propose a fix."""

# llm.complete(v1_prompt)
# response was ~600 tokens of english. unusable.

## v2: ask for json, no schema

model returned valid-ish json half the time. other half: trailing prose after the closing brace, or json wrapped in ```json fences. parsing was a nightmare.

lesson: "please respond in json" is not a contract.

In [ ]:
v2_prompt = f"""Diagnose this failure. Respond ONLY in JSON with keys:
category, root_cause, proposed_fix, confidence.

Failure:
{json.dumps(incident, indent=2)}"""

# llm.complete(v2_prompt)
# success rate maybe 60%. moving on.

## v3: json mode + pydantic schema

this is the version that's going in `sentinel/agent/graph.py` next week. response_format=json_object plus pydantic validation on our side. when the model produces invalid json the wrapper raises LLMError and the agent treats that as a model failure (i.e. it's allowed to retry the prompt once with a stricter system message).

**TODO**: add the few-shot examples once the chaos harness has produced a handful of real incidents to draw from. canned examples are worse than no examples — they over-anchor.

In [ ]:
class Diagnosis(BaseModel):
    category: str = Field(description="one of: upstream_outage, schema_drift, data_quality, infra, unknown")
    root_cause: str = Field(description="one sentence")
    proposed_fix: str
    confidence: float = Field(ge=0, le=1)
    can_auto_remediate: bool = Field(description="only true if the proposed fix is on the allowlist (retry, slip-window, coerce-to-string)")

system = (
    "You triage data pipeline failures for a self-healing pipeline. "
    "Be terse. If you're not sure, say category=unknown and confidence<0.5. "
    "can_auto_remediate must be false unless the fix is one of: "
    "(retry-with-backoff), (partition-window-slip), (coerce-to-string)."
)

v3_prompt = f"Failure record:\n{json.dumps(incident, indent=2)}\n\nReturn JSON matching the schema."

# pre-load mock with a sane response so this cell runs without an API key
llm.queue({
    "category": "upstream_outage",
    "root_cause": "TLC CloudFront returned 503 for a single partition; transient.",
    "proposed_fix": "retry-with-backoff",
    "confidence": 0.78,
    "can_auto_remediate": True,
})

resp = llm.complete(v3_prompt, system=system, json_schema=Diagnosis)
resp.parsed

## things still bothering me

- the model is happy to claim `can_auto_remediate=true` for fixes that aren't actually on the allowlist. need an explicit verify step in the graph (week 10).
- confidence is meaningless. it just outputs ~0.85 most of the time. consider replacing with a calibrated logit eventually but not soon.
- haven't tried anthropic yet. groq's llama is good enough for now.

## things I tried and decided against

- chain-of-thought. with json mode it makes the model output `"reasoning": "..."` which then dominates token cost. dropped it.
- few-shot with three canned examples. over-anchored on the first example's category every time. maybe revisit with retrieved real-incident examples (week 9 will pull these from qdrant).